# S6E9: Does Breaking Ties help?

I scored **0.94651** with the pinned [jazivxt reference](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline). In this version, I reproduce the rank-band recipe from [megayak's Reading The Public Split](https://www.kaggle.com/code/megayak/s6e9-0-94656-reading-the-public-split), whose notebook reports **0.94656**.

**I have not yet scored this new candidate.** The published bands were selected using public-leaderboard feedback. This is a public-score experiment, not evidence of a better model or improved private-leaderboard performance. I did not discover these bands or train a new model.

| Submission | Public LB | Evidence |
|---|---:|---|
| My previous public output | 0.94650 | I confirmed this score |
| My pinned reference, submission 56267408 | **0.94651** | I confirmed this score |
| Published rank-band recipe | 0.94656 | Reported by megayak |
| My reproduction in this version | Pending | I have not submitted this candidate yet |

The title stays the same, but the distinction matters: **swapping rank bands changes the order of distinct predictions; it is not merely breaking ties**. I preserve the 0.94651 baseline alongside the new candidate for comparison.


## 1. Inputs and mode

Keep the same competition dataset and **`s6e9_reference_094651_inputs.zip`** used in my previous version. No new input ZIP is required. CPU is sufficient and Internet can be off.

The ZIP contains the exact `anchor_56267408.csv`. Alternatively, attach jazivxt's dataset with `submission_latest_best.csv`; its contents must match the pinned hash below.

The default `public_band_swaps` exports the new experimental recipe. Set `MODE = "baseline"` to reproduce my confirmed 0.94651 file unchanged. Both files are exported in either mode.


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from IPython.display import FileLink, display

MODE = 'public_band_swaps'  # Alternative: 'baseline'
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## 2. Verify the scored file and its IDs

This check pins the input to submission **56267408**. It prevents a later dataset update from silently changing this experiment.


In [ ]:
EXPECTED_SHA256 = '1f40cec214646ba50eac6d5307d41c6aca82fe7acc38e2501399980c62d4bbe7'

def find_one(name):
    matches = list(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}, found {len(matches)}. Check attached datasets.')
    return matches[0]

pinned = list(INPUT_ROOT.rglob('anchor_56267408.csv'))
if len(pinned) > 1:
    raise ValueError('Multiple pinned reference files found. Attach only one copy.')
reference_path = pinned[0] if pinned else find_one('submission_latest_best.csv')
actual_hash = hashlib.sha256(reference_path.read_bytes()).hexdigest()
if actual_hash != EXPECTED_SHA256:
    raise ValueError('Reference version differs. Attach the pinned 0.94651 input ZIP.')

sample = pd.read_csv(find_one('sample_submission.csv'))
submission = pd.read_csv(reference_path, float_precision='round_trip')
assert submission.columns.tolist() == ['id', 'Will_Buy_EV']
assert len(submission) == len(sample) == 286571
assert submission.id.is_unique
np.testing.assert_array_equal(submission.id.to_numpy(), sample.id.to_numpy())
assert np.isfinite(submission.Will_Buy_EV).all()
assert submission.Will_Buy_EV.between(0, 1).all()
print('Verified: exact scored-file hash, ID order, 286,571 rows and valid probabilities.')


## 3. Reproduce the published rank-band swaps

I first convert the pinned reference predictions to percentile ranks. Within each interval below, the lower half moves above the upper half, preserving order within each half. All masks use the **original** ranks, matching the published implementation.

| Original rank interval | Half-width |
|---|---:|
| 0.54–0.58 | 0.02 |
| 0.20–0.26 | 0.03 |
| 0.58–0.62 | 0.02 |
| 0.47–0.53 | 0.03 |

The final ordinal ranking matches the author's recipe. If adjusted values tie, ordinal ranking follows row order; that adds no learned signal. I apply no extra rounding, neural blending or boundary rules.


In [ ]:
PUBLISHED_BANDS = [(0.54, 0.04), (0.20, 0.06), (0.58, 0.04), (0.47, 0.06)]

def published_band_swaps(probabilities):
    original_ranks = rankdata(probabilities, method='average') / len(probabilities)
    adjusted_ranks = original_ranks.copy()
    for start, width in PUBLISHED_BANDS:
        half = width / 2
        lower = (original_ranks >= start) & (original_ranks < start + half)
        upper = (original_ranks >= start + half) & (original_ranks < start + width)
        adjusted_ranks[lower] += half
        adjusted_ranks[upper] -= half
    return rankdata(adjusted_ranks, method='ordinal') / len(adjusted_ranks)

if MODE not in {'baseline', 'public_band_swaps'}:
    raise ValueError('Choose baseline or public_band_swaps.')

baseline_prediction = submission.Will_Buy_EV.to_numpy()
candidate_prediction = published_band_swaps(baseline_prediction)
assert np.isfinite(candidate_prediction).all()
assert ((candidate_prediction >= 0) & (candidate_prediction <= 1)).all()
assert len(np.unique(candidate_prediction)) == len(candidate_prediction)

baseline_order = rankdata(baseline_prediction, method='ordinal')
candidate_order = rankdata(candidate_prediction, method='ordinal')
changed_rows = int(np.count_nonzero(baseline_order != candidate_order))
print(f'Rows whose rank changed: {changed_rows:,} / {len(submission):,}')


## 4. Download the selected submission

I export three files:

- **`submission.csv`**: the file selected by `MODE`.
- **`submission_public_band_swaps.csv`**: the new, locally unscored reproduction.
- **`submission_baseline_094651.csv`**: my confirmed reference, copied unchanged.

The new candidate's 0.94656 is an author-reported result, not my measured score. I keep the baseline for comparison and make no private-score claim.


In [ ]:
baseline_path = OUTPUT_ROOT/'submission_baseline_094651.csv'
candidate_path = OUTPUT_ROOT/'submission_public_band_swaps.csv'
output_path = OUTPUT_ROOT/'submission.csv'

shutil.copyfile(reference_path, baseline_path)
assert hashlib.sha256(baseline_path.read_bytes()).hexdigest() == EXPECTED_SHA256
candidate = sample[['id']].copy()
candidate['Will_Buy_EV'] = candidate_prediction
candidate.to_csv(candidate_path, index=False)
shutil.copyfile(candidate_path if MODE == 'public_band_swaps' else baseline_path, output_path)

manifest = {
    'mode': MODE,
    'reference_sha256': EXPECTED_SHA256,
    'reference_submission_id': '56267408',
    'my_baseline_public_lb': 0.94651,
    'recipe_author_reported_public_lb': 0.94656,
    'my_candidate_public_lb': None,
    'recipe_source': 'https://www.kaggle.com/code/megayak/s6e9-0-94656-reading-the-public-split',
    'reference_source': 'https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline',
    'bands': PUBLISHED_BANDS,
    'changed_rank_rows': changed_rows,
    'output_sha256': hashlib.sha256(output_path.read_bytes()).hexdigest(),
}
(OUTPUT_ROOT/'submission_manifest.json').write_text(json.dumps(manifest, indent=2))
print(f'Ready: {output_path} | mode={MODE} | {len(submission):,} rows')
print('My new candidate score: pending.' if MODE == 'public_band_swaps' else 'My confirmed baseline LB: 0.94651.')
for path in [output_path, candidate_path, baseline_path]:
    display(FileLink(str(path)))
display(pd.read_csv(output_path).head())
